# 导入各种库

In [1]:
import numpy as np
# 给被删掉的别名补上到内置类型
if not hasattr(np, 'object'):
    np.object = object
if not hasattr(np, 'bool'):
    np.bool = bool
if not hasattr(np, 'int'):
    np.int = int
if not hasattr(np, 'float'):
    np.float = float
if not hasattr(np, 'complex'):
    np.complex = complex
if not hasattr(np, 'bool8'):
    np.bool8 = np.bool_

/var/folders/w5/1dk0jpjj4bgb8f3fzwqfptj40000gq/T/ipykernel_7703/950813108.py:3: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, 'object'):


In [2]:
# %matplotlib inline
import matplotlib, matplotlib.pyplot as plt
print(matplotlib.get_backend())
from backtrader_plotting import Bokeh
from backtrader_plotting.schemes import Tradimo
from IPython.display import display
import jupyterlab, sys
print(jupyterlab.__version__, " @ ", sys.executable)


module://matplotlib_inline.backend_inline


Loading BokehJS ...

4.4.6  @  /Library/Developer/CommandLineTools/usr/bin/python3


In [3]:
import backtrader as bt
import backtrader.indicators as btind # 导入策略分析模块
import pandas as pd
import tushare as ts
import warnings
import datetime
warnings.filterwarnings("ignore")



In [4]:
# 引入自己开发的模块
from pathlib import Path
import sys
root = Path.cwd()
if not (root / 'common').exists():      # 如果当前目录没有 common/
    sys.path.insert(0, str(root.parent))  # 把父目录放到 import 路径最前

from common.utils import get_engine, AShareCommission, AShareSizer
from DataFetch.stock_basic import *

# 从本地mysql取数据

In [5]:
engine = get_engine()

In [6]:

ts_code = '000001.SZ'
query = f"""
SELECT * FROM daily_kline
WHERE ts_code = '{ts_code}'
ORDER BY trade_date ASC
"""
df_sql1 = pd.read_sql(query, engine)
df_sql1['date'] = pd.to_datetime(df_sql1['trade_date'])

In [7]:
ts_code = '002594.SZ'
query = f"""
SELECT * FROM daily_kline
WHERE ts_code = '{ts_code}'
ORDER BY trade_date ASC
"""
df_sql2 = pd.read_sql(query, engine)
df_sql2['date'] = pd.to_datetime(df_sql2['trade_date'])

In [27]:
df_sql1.head()

,ts_code,trade_date,open,high,low,close,pre_close,change,pct_chg,vol,amount,date
0,000001.SZ,20100104,24.52,24.58,23.68,23.71,24.37,-0.66,-2.71,241923.0,580250.0,2010-01-04
1,000001.SZ,20100105,23.75,23.90,22.75,23.30,23.71,-0.41,-1.73,556500.0,1293480.0,2010-01-05
2,000001.SZ,20100106,23.25,23.25,22.72,22.90,23.30,-0.40,-1.72,412143.0,944454.0,2010-01-06
3,000001.SZ,20100107,22.90,23.05,22.40,22.65,22.90,-0.25,-1.09,355337.0,804166.0,2010-01-07
4,000001.SZ,20100108,22.50,22.75,22.35,22.60,22.65,-0.05,-0.22,288543.0,650667.0,2010-01-08


# 从tushare取数据

In [8]:
# from DataFetch.stock_basic import *

In [9]:
# stock_daily = StockDailyFetch()
# stock_daily.fetch_data(name = "000001.SZ 数据", ts_code="000001.SZ", start_date="20100901", end_date = "20101230")
# df_ts000001 = stock_daily.data
# df_ts000001['date'] = pd.to_datetime(df_ts000001['trade_date'])

In [10]:
# stock_daily = StockDailyFetch()
# stock_daily.fetch_data(name = "600519.SH 数据", ts_code="600519.SH", start_date="20100901", end_date = "20101230")
# df_ts600519 = stock_daily.data
# df_ts600519['date'] = pd.to_datetime(df_ts600519['trade_date'])

# 简单策略demo

In [11]:
class TS_Data(bt.feeds.PandasData):
    # 要添加的线 (初始 'close', 'low', 'high', 'open', 'volume', 'openinterest', 'datetime')
    lines = ('pre_close', 'change', 'amount', 'extra') 
    # 设置 line 在数据源上的列位置
    # -1 表示自动按列名匹配数据, None表示不存在（datetime则表示在index）, string直接写数据中的列名
    params = (
        ('fromdate', datetime.datetime(2022,4,1)),
        ('todate', datetime.datetime(2022,8,8)),
        ('nullvalue', 0.0),
        ('dtformat', ('%Y-%m-%d')),
        ('datetime', "date"),
        ('open', "open"),
        ('high', "high"),
        ('low', "low"),
        ('close', "close"),
        ('pre_close', "pre_close"),
        ('change', "change"),
        ('openinterest', "pct_chg"),
        ('volume', "vol"),
        ('amount', "amount"),
        ('extra', -1)
    )
    # def _load(self):
    #     # 调用原始的 _load 方法
    #     super()._load()

    #     # 添加你的自定义逻辑
    #     # _load方法是在每一个数据点被加载时调用的，所以你可以在这里添加你的自定义逻辑。
    #     # 你可以访问self.p.dataname来获取原始的Pandas DataFrame，然后从中提取你需要的数据。
    #     self.lines.extra[0] = self.p.dataname['extra'][self._idx]

    #     return True

In [12]:
class StockCommission(bt.CommInfoBase):
    params = (
        ('stocklike', True), # 指定为股票模式
        ('commtype', bt.CommInfoBase.COMM_PERC), # 使用百分比费用模式
        ('percabs', True), # commission 不以 % 为单位
        ('stamp_duty', 0.001), # 印花税默认为 0.1%
        ('commission', 0.0001), # 交易佣金默认为 0.01%
     ) 
    
    # 自定义费用计算公式
    def _getcommission(self, size, price, pseudoexec):
        if size > 0: # 买入时，只考虑佣金
            return abs(size) * price * self.p.commission
        elif size < 0: # 卖出时，同时考虑佣金和印花税
            return abs(size) * price * (self.p.commission + self.p.stamp_duty)
        else:
            return 0

In [13]:
# 初始化 交易
cerebro = bt.Cerebro()
# 设置初始资金
cerebro.broker.setcash(50000.0)
cerebro.broker.getcash() # 获取当前可用资金

# 设置交易费用
comminfo = StockCommission()                   
cerebro.broker.addcommissioninfo(comminfo)

# 通过调用 brokers 的 set_slippage_perc 方法设置百分比滑点
cerebro.broker.set_slippage_perc(perc=0.0001)



# data_ts_600519 = TS_Data(dataname=df_ts600519, fromdate=datetime.datetime(2010,9,1),
#                 todate=datetime.datetime(2010,12,30), nullvalue = None)
# data_ts_000001 = TS_Data(dataname=df_ts000001, fromdate=datetime.datetime(2010,9,1),
#                 todate=datetime.datetime(2010,12,30), nullvalue = None)

data_sql1 = TS_Data(dataname=df_sql1, fromdate=datetime.datetime(2025,1,1),todate=datetime.datetime(2025,8,8))
data_sql2 = TS_Data(dataname=df_sql2, fromdate=datetime.datetime(2025,1,1),todate=datetime.datetime(2025,8,8))
cerebro.adddata(data_sql1, name='000001')
cerebro.adddata(data_sql2, name='002594')

In [14]:


class TestStrategy(bt.Strategy):
    # self.data = self.datas[0]   self.dataX = self.datas[0]
    def __init__(self):
        self.dec = "="*8
        print("{dec} {text:^50s} {dec}".format(dec=self.dec, text="打印 self 策略本身的 lines"))
        print(self.lines.getlinealiases())
        print("{dec} {text:^50s} {dec}".format(dec=self.dec, text="打印 self.datas 第一个数据表格的 lines"))
        print(self.datas[0].lines.getlinealiases())
        print("{dec} {text:^50s} {dec}".format(dec=self.dec, text="打印 最后一天数据"))

        me1 = btind.EMA(self.datas[0].close, period=3)

        # Add a MovingAverageSimple indicator
        self.sma = bt.indicators.SimpleMovingAverage(
            self.datas[0], period=5)

        # Indicators for the plotting show
        bt.indicators.ExponentialMovingAverage(self.datas[0], period=25)
        bt.indicators.WeightedMovingAverage(self.datas[0], period=25,
                                            subplot=True)
        bt.indicators.StochasticSlow(self.datas[0])
        bt.indicators.MACDHisto(self.datas[0])
        rsi = bt.indicators.RSI(self.datas[0])
        bt.indicators.SmoothedMovingAverage(rsi, period=10)
        bt.indicators.ATR(self.datas[0], plot=False)

        

    def prenext(self):
        print("prenext ", bt.num2date(self.datas[0].datetime[0]))

    def nextstart(self):
        
        print("nextstart ", bt.num2date(self.datas[0].datetime[0]))
        self.next()
    
    def next(self):
        print("{dec} {text:^50s} {dec}".format(dec=self.dec, text="打印 当天数据"))

        print("date: {}, data1 close: {:.2f}, data2 close: {:.2f}. ".format(
                bt.num2date(self.datas[0].datetime[0]), self.datas[0].close[0], self.datas[1].close[0]))
        print('seen:', len(self.data), ' / total:', self.data.buflen())
        print('seen:', len(self.datas[0].close), ' / total:', self.datas[0].close.buflen())
        # print(self.datas[0].close[-1], "\nself.datas[0].lines.close.get(ago=0, size=2)", self.datas[0].lines.close.get(ago=0, size=2), 
        #       "\nself.datas[0].lines.close.get(ago=1, size=2)", self.datas[0].lines.close.get(ago=1, size=2),
        #       "\nself.datas[0].lines.close.get(size=3)", self.datas[0].lines.close.get(size=3),
        #       "\nself.datas[0].lines.close.get(ago=-1,size=3)", self.datas[0].lines.close.get(ago=-1,size=3))


In [15]:
cerebro.addstrategy(TestStrategy)
cerebro.run()

========                打印 self 策略本身的 lines                 ========
('datetime',)
========            打印 self.datas 第一个数据表格的 lines            ========
('close', 'low', 'high', 'open', 'volume', 'openinterest', 'datetime', 'pre_close', 'change', 'amount', 'extra')
========                     打印 最后一天数据                      ========
prenext  2025-01-02 00:00:00
prenext  2025-01-03 00:00:00
prenext  2025-01-06 00:00:00
prenext  2025-01-07 00:00:00
prenext  2025-01-08 00:00:00
prenext  2025-01-09 00:00:00
prenext  2025-01-10 00:00:00
prenext  2025-01-13 00:00:00
prenext  2025-01-14 00:00:00
prenext  2025-01-15 00:00:00
prenext  2025-01-16 00:00:00
prenext  2025-01-17 00:00:00
prenext  2025-01-20 00:00:00
prenext  2025-01-21 00:00:00
prenext  2025-01-22 00:00:00
prenext  2025-01-23 00:00:00
prenext  2025-01-24 00:00:00
prenext  2025-01-27 00:00:00
prenext  2025-02-05 00:00:00
prenext  2025-02-06 00:00:00
prenext  2025-02-07 00:00:00
prenext  2025-02-10 00:00:00
prenext  2025-02-11 00:00:00

In [16]:

# b = Bokeh(style='bar', scheme=Tradimo())  # 或 b = Bokeh() 默认也行
cerebro.plot(Bokeh(style='bar')) 

[[<backtrader_plotting.bokeh.bokeh.FigurePage at 0x116a77bb0>]]

In [17]:
class My_MACD(bt.Indicator):
    lines = ('macd', 'signal', 'histo')
    params = (('period_me1',12),
              ('period_me2', 26),
              ('period_signal', 9),)

    def __init__(self):
        me1 = btind.EMA(self.datas[0].close, period=self.p.period_me1)
        me2 = btind.EMA(self.datas[0].close, period=self.p.period_me2)
        self.l.macd = me1 - me2
        self.l.signal = btind.EMA(self.l.macd, period=self.p.period_signal)
        self.l.histo = self.l.macd - self.l.signal

class TestStrategy1(bt.Strategy):
    # self.data = self.datas[0]   self.dataX = self.datas[0]
    def __init__(self):
        self.dec = "="*8
        print("{dec} {text:^50s} {dec}".format(dec=self.dec, text="打印 self 策略本身的 lines"))
        print(self.lines.getlinealiases())
        print("{dec} {text:^50s} {dec}".format(dec=self.dec, text="打印 self.datas 第一个数据表格的 lines"))
        print(self.datas[0].lines.getlinealiases())
        print("{dec} {text:^50s} {dec}".format(dec=self.dec, text="打印 最后一天数据"))
        print(bt.num2date(self.datas[0].datetime[0]), end = ':\t')
        print(self.datas[0].close[0], end = "\t")
        print(self.datas[0].openinterest[0])

        # 
        self.ma3 = btind.SimpleMovingAverage(self.datas[0].close, period=3)
        self.ma5 = btind.SimpleMovingAverage(self.datas[0].close, period=5)
        self.macd = My_MACD(period_me1=3,period_me2=5,period_signal=2)
        
    def next(self):
        print("{dec} {text:^50s} {dec}".format(dec=self.dec, text="打印 当天数据"))
        # print("date: {}, close: {:.2f}, ma3: {:.2f}, ma5: {:.2f}, macd: {:.2f}, signal: {:.2f}, histo: {:.2f}. ".format(
        #         bt.num2date(self.datas[0].datetime[0]), self.datas[0].close[0], self.ma3[0], self.ma5[0], 
        #         self.macd.macd[0], self.macd.signal[0], self.macd.histo[0]))
        print("date: {}, data1 close: {:.2f}, data2 close: {:.2f}. ".format(
                bt.num2date(self.datas[0].datetime[0]), self.datas[0].close[0], self.datas[1].close[0]))
        if self.ma5 > self.ma3:
            print("买买买")

In [18]:
# 初始化 交易
cerebro = bt.Cerebro()
# 设置初始资金
cerebro.broker.setcash(50000.0)
cerebro.broker.getcash() # 获取当前可用资金

# 设置交易费用
comminfo = StockCommission()                   
cerebro.broker.addcommissioninfo(comminfo)

# 通过调用 brokers 的 set_slippage_perc 方法设置百分比滑点
cerebro.broker.set_slippage_perc(perc=0.0001)



# data_ts_600519 = TS_Data(dataname=df_ts600519, fromdate=datetime.datetime(2010,9,1),
#                 todate=datetime.datetime(2010,12,30), nullvalue = None)
# data_ts_000001 = TS_Data(dataname=df_ts000001, fromdate=datetime.datetime(2010,9,1),
#                 todate=datetime.datetime(2010,12,30), nullvalue = None)

data_sql1 = TS_Data(dataname=df_sql1, fromdate=datetime.datetime(2025,1,1),todate=datetime.datetime(2025,8,8))
data_sql2 = TS_Data(dataname=df_sql2, fromdate=datetime.datetime(2025,1,1),todate=datetime.datetime(2025,8,8))
cerebro.adddata(data_sql1, name='000001')
cerebro.adddata(data_sql2, name='002594')

In [19]:
cerebro.addstrategy(TestStrategy1)
cerebro.run()

========                打印 self 策略本身的 lines                 ========
('datetime',)
========            打印 self.datas 第一个数据表格的 lines            ========
('close', 'low', 'high', 'open', 'volume', 'openinterest', 'datetime', 'pre_close', 'change', 'amount', 'extra')
========                     打印 最后一天数据                      ========
2025-08-08 00:00:00:	12.4	-0.5613
========                      打印 当天数据                       ========
date: 2025-01-09 00:00:00, data1 close: 11.40, data2 close: 269.39. 
========                      打印 当天数据                       ========
date: 2025-01-10 00:00:00, data1 close: 11.30, data2 close: 266.02. 
买买买
========                      打印 当天数据                       ========
date: 2025-01-13 00:00:00, data1 close: 11.20, data2 close: 266.00. 
买买买
========                      打印 当天数据                       ========
date: 2025-01-14 00:00:00, data1 close: 11.38, data2 close: 276.30. 
买买买
========                      打印 当天数据                       ========

In [20]:
# 买卖demo

In [21]:

cerebro.plot(Bokeh(style='bar')) 

[[<backtrader_plotting.bokeh.bokeh.FigurePage at 0x116da58e0>]]

## A股模拟

In [22]:







# -------- 3) T+1 示例策略：当日买入，次日才能卖出 --------
class T1ExampleStrategy(bt.Strategy):
    params = dict(
        pfast=10, pslow=20,
    )

    def __init__(self):
        self.sma_fast = bt.ind.SMA(self.data.close, period=self.p.pfast)
        self.sma_slow = bt.ind.SMA(self.data.close, period=self.p.pslow)

        # 记录最近一次“买入完成”的交易日（用于T+1）
        self.last_buy_date = None

    def notify_order(self, order):
        if order.status in [order.Completed]:
            odt = bt.num2date(order.executed.dt).date()
            if order.isbuy():
                # 记录买入完成的日期
                self.last_buy_date = odt

    def can_sell_today(self):
        """T+1：如果今天与最近买入完成是同一天，则不能卖出。"""
        if self.position.size <= 0:
            return False  # 没有持仓无需卖
        if self.last_buy_date is None:
            return True   # 没买过，自由卖
        today = self.data.datetime.date(0)
        return self.last_buy_date < today  # 只有过了一天才可卖

    def next(self):
        # 简单金叉买、死叉卖（演示），加上 T+1 约束
        if not self.position:
            if bt.ind.CrossOver(self.sma_fast, self.sma_slow)[0] > 0:
                self.buy()  # 具体买入数量由 sizer 控制成 100 的整数倍
        else:
            # 死叉出现且满足T+1才卖
            if bt.ind.CrossOver(self.sma_fast, self.sma_slow)[0] < 0 and self.can_sell_today():
                self.sell()




In [23]:
# Create a Stratey
class TestStrategy(bt.Strategy):

    def log(self, txt, dt=None):
        ''' Logging function fot this strategy'''
        dt = dt or self.datas[0].datetime.date(0)
        print('%s, %s' % (dt.isoformat(), txt))

    def __init__(self):
        # Keep a reference to the "close" line in the data[0] dataseries
        self.dataclose = self.datas[0].close

        # To keep track of pending orders
        self.order = None

    def notify_order(self, order):
        if order.status in [order.Submitted, order.Accepted]:
            # Buy/Sell order submitted/accepted to/by broker - Nothing to do
            return

        # Check if an order has been completed
        # Attention: broker could reject order if not enough cash
        if order.status in [order.Completed]:
            if order.isbuy():
                self.log('BUY EXECUTED, %.2f' % order.executed.price)
            elif order.issell():
                self.log('SELL EXECUTED, %.2f' % order.executed.price)

            self.bar_executed = len(self)

        elif order.status in [order.Canceled]:
            self.log('Order Canceled')
        elif order.status in [order.Margin]:
            self.log('Order Margin')
        elif order.status in [order.Rejected]:
            self.log('Order Rejected')

        # Write down: no pending order
        self.order = None

    def next(self):
        # Simply log the closing price of the series from the reference
        self.log('Close, %.2f' % self.dataclose[0])

        # Check if an order is pending ... if yes, we cannot send a 2nd one
        if self.order:
            return

        # Check if we are in the market
        if not self.position:

            # Not yet ... we MIGHT BUY if ...
            if self.dataclose[0] < self.dataclose[-1]:
                    # current close less than previous close

                    if self.dataclose[-1] < self.dataclose[-2]:
                        # previous close less than the previous close

                        # BUY, BUY, BUY!!! (with default parameters)
                        self.log('BUY CREATE, %.2f' % self.dataclose[0])

                        # Keep track of the created order to avoid a 2nd order
                        self.order = self.buy()

        else:

            # Already in the market ... we might sell
            if len(self) >= (self.bar_executed + 5):
                # SELL, SELL, SELL!!! (with all possible default parameters)
                self.log('SELL CREATE, %.2f' % self.dataclose[0])

                # Keep track of the created order to avoid a 2nd order
                self.order = self.sell()

In [24]:
# 初始化 交易
cerebro = bt.Cerebro()
# 设置初始资金
cerebro.broker.setcash(10000000.0)
cerebro.broker.getcash() # 获取当前可用资金

# 加入A股佣金模型（可按需调整参数）
comminfo = AShareCommission(
    commission=0.0003,      # 券商佣金 0.03%
    stamp_duty=0.0005,      # 卖出印花税 0.05%
    min_commission=5.0,     # 最低佣金 5 元
    transfer_fee=0.0        # 如需上证过户费可设为0.00002
)
cerebro.broker.addcommissioninfo(comminfo)

# A股手数控制（买入 100 股的整数倍）
cerebro.addsizer(AShareSizer, lot_size=100, cash_keep=0.8)

# 通过调用 brokers 的 set_slippage_perc 方法设置百分比滑点
cerebro.broker.set_slippage_perc(perc=0.0001)



# data_ts_600519 = TS_Data(dataname=df_ts600519, fromdate=datetime.datetime(2010,9,1),
#                 todate=datetime.datetime(2010,12,30), nullvalue = None)
# data_ts_000001 = TS_Data(dataname=df_ts000001, fromdate=datetime.datetime(2010,9,1),
#                 todate=datetime.datetime(2010,12,30), nullvalue = None)

data_sql1 = TS_Data(dataname=df_sql1, fromdate=datetime.datetime(2025,1,1),todate=datetime.datetime(2025,8,8))
data_sql2 = TS_Data(dataname=df_sql2, fromdate=datetime.datetime(2025,1,1),todate=datetime.datetime(2025,8,8))
cerebro.adddata(data_sql1, name='000001')
# cerebro.adddata(data_sql2, name='002594')

In [25]:
cerebro.addstrategy(TestStrategy)
cerebro.run()

2025-01-02, Close, 11.43
2025-01-02, BUY CREATE, 11.43
2025-01-03, BUY EXECUTED, 11.44
2025-01-03, Close, 11.38
2025-01-06, Close, 11.44
2025-01-07, Close, 11.51
2025-01-08, Close, 11.50
2025-01-09, Close, 11.40
2025-01-10, Close, 11.30
2025-01-10, SELL CREATE, 11.30
2025-01-13, SELL EXECUTED, 11.25
2025-01-13, Close, 11.20
2025-01-13, BUY CREATE, 11.20
2025-01-14, BUY EXECUTED, 11.20
2025-01-14, Close, 11.38
2025-01-15, Close, 11.48
2025-01-16, Close, 11.57
2025-01-17, Close, 11.45
2025-01-20, Close, 11.42
2025-01-21, Close, 11.33
2025-01-21, SELL CREATE, 11.33
2025-01-22, SELL EXECUTED, 11.32
2025-01-22, Close, 11.09
2025-01-22, BUY CREATE, 11.09
2025-01-23, BUY EXECUTED, 11.17
2025-01-23, Close, 11.32
2025-01-24, Close, 11.34
2025-01-27, Close, 11.47
2025-02-05, Close, 11.37
2025-02-06, Close, 11.36
2025-02-07, Close, 11.38
2025-02-07, SELL CREATE, 11.38
2025-02-10, SELL EXECUTED, 11.38
2025-02-10, Close, 11.43
2025-02-11, Close, 11.42
2025-02-12, Close, 11.42
2025-02-13, Close, 11.

In [26]:


cerebro.plot(Bokeh(style='bar')) 

[[<backtrader_plotting.bokeh.bokeh.FigurePage at 0x116921040>]]

In [ ]:
-202